### 복숭아 재배 농가 위치 지점ID 매핑

In [3]:
# pip install pyproj
import os
import numpy as np
import pandas as pd
from pyproj import Transformer

FILES = {
    "PEACH": "/Users/doyoung-gil/연구실/데이터/복숭아/복숭아 재배 농가 데이터.csv",
}

CELL_M = 30
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

for crop, path in FILES.items():
    if not os.path.exists(path):
        print(f"[건너뜀] {crop}: {path} 없음")
        continue

    # 1) 파일 로드
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]  # 컬럼명 공백 제거(안전)

    # 2) decimalLongitude/Latitude -> 좌표-경도/위도 로 '이름 변경' (원본 컬럼은 사라짐)
    df = df.rename(columns={
        "decimalLongitude": "좌표-경도",
        "decimalLatitude": "좌표-위도"
    })

    # 컬럼 존재 확인
    if "좌표-경도" not in df.columns or "좌표-위도" not in df.columns:
        raise SystemExit(f"필수 컬럼이 없습니다. columns={list(df.columns)}")

    # 3) 좌표 숫자화 + 유효 마스크
    df["좌표-경도"] = pd.to_numeric(df["좌표-경도"], errors="coerce")
    df["좌표-위도"] = pd.to_numeric(df["좌표-위도"], errors="coerce")
    valid = df["좌표-경도"].notna() & df["좌표-위도"].notna()

    # 4) 좌표 변환(유효 행만)
    x = np.full(len(df), np.nan)
    y = np.full(len(df), np.nan)
    if valid.any():
        xv, yv = tf.transform(
            df.loc[valid, "좌표-경도"].values,
            df.loc[valid, "좌표-위도"].values
        )
        x[valid], y[valid] = xv, yv

    # 5) 30m 격자 index → 지점ID 생성
    ix = pd.Series(np.floor_divide(x, CELL_M)).astype("Int64")
    iy = pd.Series(np.floor_divide(y, CELL_M)).astype("Int64")

    df["지점ID"] = pd.NA
    df.loc[valid, "지점ID"] = ix[valid].astype(str).str.cat(iy[valid].astype(str), sep="_")

    # 6) 지점ID를 맨 앞으로
    cols = ["지점ID"] + [c for c in df.columns if c != "지점ID"]
    df = df[cols]

    # 7) 저장
    base, ext = os.path.splitext(path)
    out_path = f"{base}_with_siteid{ext}"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {crop}: {out_path} | rows={len(df):,}, 지점ID 생성={df['지점ID'].notna().sum():,}")


[저장] PEACH: /Users/doyoung-gil/연구실/데이터/복숭아/복숭아 재배 농가 데이터_with_siteid.csv | rows=77, 지점ID 생성=77


### 지점 마스터 만들기

In [4]:
# make_site_master_from_leafblast_fix.py
from pathlib import Path

OBS_DIR = Path("/Users/doyoung-gil/연구실/데이터/복숭아")
BASE    = OBS_DIR / "복숭아 재배 농가 데이터_with_siteid.csv"
OUT     = OBS_DIR / "site_master.csv"

def read_df(path: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "euc-kr", "cp949", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, dtype=str, low_memory=False)

df = read_df(BASE)

# 컬럼 이름 정리
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={c: c.replace(" ", "") for c in df.columns})

# 필요한 컬럼 찾기
id_col  = next((c for c in df.columns if c.lower() in ("지점id","siteid","site_id","지점")), None)
lat_col = next((c for c in df.columns if "위도" in c), None)
lon_col = next((c for c in df.columns if "경도" in c), None)
if not all([id_col, lat_col, lon_col]):
    raise SystemExit(f"필수 컬럼을 찾지 못했습니다: id={id_col}, lat={lat_col}, lon={lon_col}")

sub = df[[id_col, lat_col, lon_col]].copy()
sub.columns = ["지점ID", "좌표-위도", "좌표-경도"]

# 지점ID는 문자열 유지
sub["지점ID"] = sub["지점ID"].astype(str).str.strip()

# 좌표만 숫자화
sub["좌표-위도"] = pd.to_numeric(sub["좌표-위도"], errors="coerce")
sub["좌표-경도"] = pd.to_numeric(sub["좌표-경도"], errors="coerce")

# 유효값만
sub = sub[(sub["지점ID"] != "") & sub["좌표-위도"].notna() & sub["좌표-경도"].notna()]

# 지점별 대표 좌표(반올림 후 중앙값)
master = (sub.round({"좌표-위도": 6, "좌표-경도": 6})
            .groupby("지점ID", as_index=False)
            .agg({"좌표-위도": "median", "좌표-경도": "median"}))

master.to_csv(OUT, index=False, encoding="utf-8-sig", float_format="%.6f")
print(f"[저장] {OUT} (지점 수={len(master):,})")
print(master.head())


[저장] /Users/doyoung-gil/연구실/데이터/복숭아/site_master.csv (지점 수=77)
          지점ID    좌표-위도     좌표-경도
0  32112_58454  35.7785  127.0947
1  32262_58780  35.8668  127.1441
2  32550_61437  36.5856  127.2374
3  32623_61483  36.5981  127.2619
4  32726_61027  36.4749  127.2968


### 이웃 관측소 마스터 만들기

In [6]:
# build_neighbors_master_csv.py
# -*- coding: utf-8 -*-
import numpy as np

# ===== 설정 =====
EARTH_R   = 6371000.0     # 지구 반경(m)
IDW_POWER = 1.5           # IDW 가중치 지수 (w = 1/d^p)
K_NEIGH   = 3             # 지점ID당 이웃 관측소 개수

WEATHER_DIR = Path("/Users/doyoung-gil/연구실/데이터/기상데이터/ASOS+AWS/GDD")
SITE_MASTER = Path("/Users/doyoung-gil/연구실/데이터/복숭아/site_master.csv")
OUT_CSV     = Path("/Users/doyoung-gil/연구실/데이터/복숭아/neighbors_master.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

DATE_COL   = "일시"
DATE_START = pd.Timestamp("2010-01-01")
DATE_END   = pd.Timestamp("2025-01-19")


# ===== 함수 =====
def hv(lat1, lon1, lat2, lon2):
    """벡터화 하버사인 거리(m)"""
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))

def read_weather_stations() -> pd.DataFrame:
    """기상 파일들에서 관측소 좌표만 모아서 고유 목록 반환."""
    parts = []
    for y in range(DATE_START.year, DATE_END.year + 1):
        p = WEATHER_DIR / f"ASOS_AWS_{y}_with_GDD10.csv"
        if not p.exists():
            continue
        w = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
        # 기간 필터(있을 때만)
        if DATE_COL in w.columns:
            w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
            w = w[(w[DATE_COL] >= DATE_START) & (w[DATE_COL] <= DATE_END)]
        # 좌표/ID 숫자화
        for c in ["지점", "위도", "경도"]:
            if c in w.columns:
                w[c] = pd.to_numeric(w[c], errors="coerce")
        parts.append(w[["지점", "위도", "경도"]])
    if not parts:
        raise SystemExit("관측소 좌표를 읽을 수 있는 기상 파일이 없습니다.")
    stn = pd.concat(parts, ignore_index=True).dropna().drop_duplicates()
    if stn.empty:
        raise SystemExit("관측소 좌표가 비어 있습니다.")
    return stn

def main():
    # 지점 마스터 로드 (지점ID는 문자열 유지)
    site = pd.read_csv(SITE_MASTER, dtype={"지점ID": str})
    site = site.dropna(subset=["좌표-위도", "좌표-경도"]).copy()
    site["지점ID"] = site["지점ID"].astype(str).str.strip()
    site = site[site["지점ID"] != ""].drop_duplicates(subset=["지점ID"])

    # 관측소 목록 로드
    stn = read_weather_stations()
    stn_lat = stn["위도"].to_numpy()
    stn_lon = stn["경도"].to_numpy()
    stn_ids = stn["지점"].to_numpy()

    rows = []
    for sid, slat, slon in site[["지점ID", "좌표-위도", "좌표-경도"]].itertuples(index=False):
        d = hv(float(slat), float(slon), stn_lat, stn_lon)       # 거리(m)
        k = min(K_NEIGH, len(stn_ids))
        order = np.argsort(d)[:k]
        d_k, ids_k = d[order], stn_ids[order]
        # 가중치(0m 보호)
        if np.isclose(d_k[0], 0.0):
            w = np.zeros_like(d_k); w[0] = 1.0
        else:
            w = 1.0 / np.power(np.maximum(d_k, 1.0), IDW_POWER)
        for r, (stn_id, dist, w_raw) in enumerate(zip(ids_k, d_k, w), start=1):
            rows.append((sid, int(stn_id), r, float(dist), float(w_raw)))

    neigh = pd.DataFrame(rows, columns=["지점ID", "지점", "rank", "거리_m", "w_raw"])
    neigh = neigh.sort_values(["지점ID", "rank"]).drop_duplicates(["지점ID", "rank"])

    # CSV 저장
    neigh.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    # 간단 요약
    cnt_by_site = neigh.groupby("지점ID")["rank"].max()
    print(f"[저장] {OUT_CSV}  rows={len(neigh):,},  지점수={cnt_by_site.size:,}")
    print("지점별 이웃개수 분포:")
    print(cnt_by_site.value_counts().sort_index())

if __name__ == "__main__":
    main()


[저장] /Users/doyoung-gil/연구실/데이터/복숭아/neighbors_master.csv  rows=231,  지점수=77
지점별 이웃개수 분포:
rank
3    77
Name: count, dtype: int64


In [ ]:
# join_weather_to_peach_sites.py
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ======================= 설정 =======================
DATE_COL   = "일시"
DATE_START = pd.Timestamp("2010-01-01")
DATE_END   = pd.Timestamp("2025-01-31")

# 입력: 복숭아 농가 지점 파일(지점ID 있어야 함)
SITE_CSV   = "/Users/doyoung-gil/연구실/데이터/복숭아/복숭아 재배 농가 데이터_with_siteid.csv"

# 기상(연도별)
WEATHER_DIR = "/Users/doyoung-gil/연구실/데이터/기상데이터/ASOS+AWS/GDD"   # ASOS_AWS_YYYY_with_GDD10.csv

# neighbors_master (지점ID -> 이웃 관측소/가중치)
NEIGH_CSV   = "/Users/doyoung-gil/연구실/데이터/복숭아/neighbors_master.csv"

# 출력
OUT_PATH    = "/Users/doyoung-gil/연구실/데이터/복숭아/peach_sites_weather_joined_2010_2025.csv"

# 결측 보강 방식
FILL_METHOD = "nearest"   # "nearest" or "idw"

# 기상 컬럼 표준화 매핑
WX_NAME_MAP = {
    "일강수량": ["일강수량(mm)", "일강수량"],
    "최고기온": ["최고기온(°C)", "최고기온(℃)", "최고기온"],
    "최저기온": ["최저기온(°C)", "최저기온(℃)", "최저기온"],
    "평균기온": ["평균기온(°C)", "평균기온(℃)", "평균기온"],
    "최대 풍속": ["최대 풍속(m/s)", "최대풍속(m/s)", "최대 풍속", "최대풍속",
               "최대 순간 풍속(m/s)", "최대순간풍속(m/s)"],
    "평균 풍속": ["평균 풍속(m/s)", "평균풍속(m/s)", "평균 풍속", "평균풍속"],
}

# 붙일 기상 feature 목록
AGG_COLS = ["일강수량", "최고기온", "최대 풍속", "최저기온", "평균 풍속", "평균기온", "DD10", "GDD10_cum"]

# ======================= 유틸 =======================
def load_weather_year(y: int) -> pd.DataFrame | None:
    p = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}_with_GDD10.csv")
    if not os.path.exists(p):
        return None
    w = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
    w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
    for c in ["지점", "위도", "경도"]:
        if c in w.columns:
            w[c] = pd.to_numeric(w[c], errors="coerce")
    if {"지점", DATE_COL}.issubset(w.columns):
        w = w.drop_duplicates(subset=["지점", DATE_COL]).sort_values(["지점", DATE_COL])
    return w

def load_weather_range(start_ts, end_ts) -> pd.DataFrame:
    parts = []
    for y in range(start_ts.year, end_ts.year + 1):
        df = load_weather_year(y)
        if df is not None and not df.empty:
            parts.append(df)
    if not parts:
        raise FileNotFoundError("기간 내 기상 파일을 찾지 못했습니다.")
    wx = pd.concat(parts, ignore_index=True)
    wx = wx[(wx[DATE_COL] >= start_ts) & (wx[DATE_COL] <= end_ts)]
    return wx

def pick_first(df: pd.DataFrame, candidates: list):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def standardize_weather_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for std, cands in WX_NAME_MAP.items():
        src = pick_first(out, cands)
        out[std] = pd.to_numeric(out[src], errors="coerce") if src else np.nan

    # DD/GDD 컬럼명 보정(파일마다 이름 다르면 여기 추가)
    if "DD10" not in out.columns and "DD" in out.columns:
        out.rename(columns={"DD": "DD10"}, inplace=True)
    if "GDD10_cum" not in out.columns and "GDD" in out.columns:
        out.rename(columns={"GDD": "GDD10_cum"}, inplace=True)

    # 없으면 NaN 컬럼 생성
    for c in ["DD10", "GDD10_cum"]:
        if c not in out.columns:
            out[c] = np.nan

    return out

def aggregate_weather_by_neighbors(wx_all: pd.DataFrame,
                                   neighbors: pd.DataFrame,
                                   date_col: str,
                                   agg_cols: list,
                                   method: str = "idw") -> pd.DataFrame:
    """
    neighbors: ['지점ID','지점','rank','거리_m','w_raw']
    return: ['지점ID', date_col] + agg_cols  (전 기간 매일)
    """
    site_ids = neighbors["지점ID"].astype(str).drop_duplicates().to_list()
    stn_ids  = wx_all["지점"].drop_duplicates().to_list()

    site_idx = {sid: i for i, sid in enumerate(site_ids)}
    stn_idx  = {sid: j for j, sid in enumerate(stn_ids)}

    n_sites = len(site_ids)
    dates   = pd.date_range(DATE_START, DATE_END, freq="D")
    n_days  = len(dates)

    r_idx = neighbors["지점ID"].map(site_idx).to_numpy()
    c_idx = neighbors["지점"].map(stn_idx).to_numpy()
    w_raw = neighbors["w_raw"].astype(float).to_numpy()

    W = csr_matrix((w_raw, (r_idx, c_idx)), shape=(n_sites, len(stn_ids)))

    out = pd.DataFrame({
        "지점ID": np.repeat(site_ids, n_days),
        date_col: np.tile(dates.values, n_sites)
    })

    for col in agg_cols:
        mat = (wx_all.pivot(index=date_col, columns="지점", values=col)
                     .reindex(index=dates, columns=stn_ids))
        X = mat.to_numpy(dtype=float)                  # (days x stations)
        valid = np.isfinite(X).astype(float)
        X0 = np.nan_to_num(X, nan=0.0)

        if method == "idw":
            N = W.dot(X0.T).astype(float)              # (sites x days)
            D = W.dot(valid.T).astype(float)
            R = np.divide(N, D, out=np.full_like(N, np.nan), where=(D > 0))
            out[col] = R.reshape(n_sites * n_days, order="C")

        elif method == "nearest":
            piv = (neighbors.sort_values(["지점ID", "rank"])
                            .drop_duplicates(["지점ID", "rank"])
                            .pivot(index="지점ID", columns="rank", values="지점")
                            .reindex(index=site_ids))
            idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)
            K = idx_by_rank.shape[1]
            R = np.full((n_days, n_sites), np.nan, dtype=float)

            for r in range(K):
                idx_r = idx_by_rank[:, r]
                mask = np.isfinite(idx_r)
                if not mask.any():
                    continue
                cols_r = idx_r.copy()
                cols_r[~mask] = 0
                vals_r = X[:, cols_r.astype(int)]
                vals_r[:, ~mask] = np.nan
                fill = np.isnan(R) & np.isfinite(vals_r)
                R[fill] = vals_r[fill]

            out[col] = R.T.reshape(n_sites * n_days, order="C")
        else:
            raise ValueError("method must be 'idw' or 'nearest'")

    return out

# ======================= 메인 =======================
def main():
    # 1) site 파일 로드 (지점ID 목록)
    sites = pd.read_csv(SITE_CSV, encoding="utf-8-sig", dtype=str, low_memory=False)
    sites.columns = [c.strip() for c in sites.columns]
    if "지점ID" not in sites.columns:
        raise SystemExit(f"SITE_CSV에 '지점ID' 컬럼이 없습니다. columns={list(sites.columns)}")
    site_ids = pd.Series(sites["지점ID"].astype(str).str.strip().dropna().unique())
    if site_ids.empty:
        raise SystemExit("지점ID가 비어 있습니다.")

    # 2) neighbors 로드 (해당 지점ID만)
    neighbors_master = pd.read_csv(NEIGH_CSV, dtype={"지점ID": str})
    neighbors_master["지점ID"] = neighbors_master["지점ID"].astype(str).str.strip()
    neighbors_master = neighbors_master.sort_values(["지점ID", "rank"]).drop_duplicates(["지점ID", "rank"])

    neighbors = neighbors_master[neighbors_master["지점ID"].isin(site_ids)]
    if neighbors.empty:
        raise SystemExit("neighbors_master에서 지점ID 매칭이 실패했습니다. (neighbors가 비어있음)")

    # 3) 기상 로드 & 표준화
    wx_all = load_weather_range(DATE_START, DATE_END)
    wx_all = standardize_weather_cols(wx_all)

    # 필요한 컬럼만
    wx_use = wx_all[["지점", DATE_COL] + AGG_COLS].copy()

    # 4) (지점ID × 매일) base 만들기
    days = pd.date_range(DATE_START, DATE_END, freq="D")
    base = (pd.DataFrame({"지점ID": site_ids})
              .assign(key=1)
              .merge(pd.DataFrame({DATE_COL: days, "key": 1}), on="key")
              .drop(columns="key"))

    # 5) neighbors 기반으로 기상 결측 보강해서 지점ID별 일별 기상 생성
    wx_filled = aggregate_weather_by_neighbors(wx_use, neighbors, DATE_COL, AGG_COLS, method=FILL_METHOD)

    # 6) base에 기상 붙이기
    out = base.merge(wx_filled, on=["지점ID", DATE_COL], how="left")

    # (옵션) 단위 컬럼 복제
    out["일강수량(mm)"]   = out["일강수량"]
    out["최고기온(°C)"]   = out["최고기온"]
    out["최대 풍속(m/s)"] = out["최대 풍속"]
    out["최저기온(°C)"]   = out["최저기온"]
    out["평균 풍속(m/s)"] = out["평균 풍속"]
    out["평균기온(°C)"]   = out["평균기온"]

    # 7) 저장 (원본 site 메타도 같이 붙이고 싶으면 아래 한 줄 추가)
    # out = out.merge(sites.drop_duplicates("지점ID"), on="지점ID", how="left")

    cols = ["지점ID", DATE_COL,
            "일강수량(mm)", "최고기온(°C)", "최대 풍속(m/s)",
            "최저기온(°C)", "평균 풍속(m/s)", "평균기온(°C)",
            "DD10", "GDD10_cum"]
    out = out[cols].sort_values(["지점ID", DATE_COL]).reset_index(drop=True)

    out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig", float_format="%.1f")
    print(f"[저장] {OUT_PATH} | rows={len(out):,}, cols={len(out.columns)}")

if __name__ == "__main__":
    main()


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_58850/2292223628.py:140: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_58850/2292223628.py:140: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_58850/2292223628.py:140: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_58850/2292223628.py:140: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_58850/2292223628.py:1

[저장] /Users/doyoung-gil/연구실/데이터/복숭아/peach_sites_weather_joined_2010_2025.csv | rows=424,270, cols=10


In [10]:
# split_timeseries_by_site_simple.py
import os, glob, re
import pandas as pd

IN_DIR   = "/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples"
OUT_ROOT = "/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples"
FILE_GLOB = "joined_SAMPLE_GDD_2010_2024_PEACH.csv"  

SITE_COL = "지점ID" 

def sanitize_for_filename(s) -> str:
    """파일명에 쓸 수 있게 간단 정제(공백/특수문자 -> _)"""
    import re
    s = "NA" if pd.isna(s) else str(s).strip()
    return re.sub(r'[^0-9A-Za-z가-힣_.-]+', "_", s)

def parse_crop_name(fname: str):
    base = os.path.basename(fname)

    # 1) CROP + NAME 있는 경우: ..._{CROP}_{NAME}.csv
    m = re.match(r"joined_SAMPLE_GDD_.+?_([A-Za-z]+)_(.+)\.csv$", base)
    if m:
        return m.group(1).upper(), m.group(2)

    # 2) CROP만 있는 경우: ..._{CROP}.csv  (네 파일)
    m = re.match(r"joined_SAMPLE_GDD_.+?_([A-Za-z]+)\.csv$", base)
    if m:
        return m.group(1).upper(), "WEATHER"   # NAME 기본값

    return None, None

def main():
    files = sorted(glob.glob(os.path.join(IN_DIR, FILE_GLOB)))
    if not files:
        raise SystemExit(f"[종료] 입력 없음: {os.path.join(IN_DIR, FILE_GLOB)}")

    for fp in files:
        crop, name = parse_crop_name(fp)
        if crop is None:
            print(f"[스킵] 파일명 패턴 불일치: {fp}")
            continue

        out_dir = os.path.join(OUT_ROOT, f"{name}_by_site")
        os.makedirs(out_dir, exist_ok=True)

        df = pd.read_csv(fp, encoding="utf-8-sig")

        if SITE_COL not in df.columns:
            print(f"[스킵] '{SITE_COL}' 컬럼 없음: {fp}")
            continue

        for site, g in df.groupby(SITE_COL, dropna=False):
            if pd.isna(site):
                continue
            site_str = sanitize_for_filename(site)
            out_path = os.path.join(out_dir, f"{crop}_{name}_site-{site_str}.csv")
            g.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"[저장 완료] {name} → {out_dir}")

if __name__ == "__main__":
    main()

[저장 완료] WEATHER → /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WEATHER_by_site


### WeatherMovingAverage 구하기

In [11]:
# make_moving_windows_for_fuzzy.py

from pathlib import Path
import pandas as pd
import numpy as np
import re, os

# ===== 경로/설정 =====
IN_DIR  = Path("/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WEATHER_by_site")
OUT_DIR = Path("/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage")   # R 스크립트와 동일 폴더명
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 2010
YEAR_END   = 2024
YEARS = list(range(YEAR_START, YEAR_END + 1))
OFFS = list(range(0, 30))     # 오프셋(일) OFFS  = [0, 5, 10, 15, 20, 25]
WIN   = 30                          # 창 길이(일)
STEP  = 30                          # 창 간격(일) = 30일씩 
DAYS_CONST = 30

# 원본 열 이름 매핑
COL_DATE  = "일시"
COL_TMAX  = "최고기온(°C)"
COL_TMIN  = "최저기온(°C)"
COL_TAVG  = "평균기온(°C)"
COL_WMEAN = "평균 풍속(m/s)"
COL_WGUST = "최대 풍속(m/s)"
COL_PRCP  = "일강수량(mm)"
COL_DD10  = "DD10"          # 없거나 전부 NaN이면 tavg 기반으로 계산: max(tavg-10, 0)

def parse_site_id(fname: str) -> str:
    base = os.path.basename(fname)
    m = re.search(r"site-([^./]+)", base)  # 파일명 'site-xxxx'
    return m.group(1) if m else os.path.splitext(base)[0]

def to_num(s): 
    return pd.to_numeric(s, errors="coerce")

def window_stats(d: pd.DataFrame):
    """30일 창 집계: 평균/최댓값/합계. 보간 없음, NaN은 제외하고 계산."""
    out = {}
    # 평균기온 결측 보정: (tmax+tmin)/2 (둘 다 있을 때만)
    tmax = to_num(d.get(COL_TMAX))
    tmin = to_num(d.get(COL_TMIN))
    tavg = to_num(d.get(COL_TAVG))
    if tavg is None:
        tavg = pd.Series(index=d.index, dtype="float64")
    tavg = tavg.copy()
    fill_mask = (tmax.notna() & tmin.notna() & tavg.isna())
    tavg[fill_mask] = (tmax[fill_mask] + tmin[fill_mask]) / 2.0

    out["tavg_mean_30d"] = float(tavg.mean(skipna=True))
    out["tmin_mean_30d"] = float(tmin.mean(skipna=True))
    out["tmax_mean_30d"] = float(tmax.mean(skipna=True))

    wmean = to_num(d.get(COL_WMEAN))
    wgust = to_num(d.get(COL_WGUST))
    prcp  = to_num(d.get(COL_PRCP))

    out["wind_mean_30d"]     = float(wmean.mean(skipna=True))
    out["wind_gust_max_30d"] = float(wgust.max(skipna=True))
    out["precip_sum_30d"]    = float(prcp.sum(skipna=True))

    dd10 = to_num(d.get(COL_DD10))
    if (dd10 is None) or (isinstance(dd10, pd.Series) and dd10.notna().sum() == 0):
        # DD10 열이 없거나 전부 결측이면 tavg로 대체 계산
        dd10 = np.maximum(tavg - 10.0, 0.0)
    out["gdd10_sum_30d"] = float(pd.Series(dd10).sum(skipna=True))
    return out

def gen_windows_for_year(year: int, offset_days: int):
    """
    해당 연도: 1월 1일 + offset에서 시작해 30일 창을 30일 간격으로 '딱 12개' 생성.
    (마지막 창은 다음 해로 넘어갈 수도 있음: 요구사항대로 그대로 둠)
    """
    base = pd.Timestamp(f"{year}-01-01")
    wins = []
    for i in range(12):
        start = base + pd.Timedelta(days=offset_days + i*STEP)
        end   = start + pd.Timedelta(days=WIN-1)  # 포함 끝
        wins.append((i+1, start, end))
    return wins

def process_one(fp: Path):
    df = pd.read_csv(fp, dtype={"지점ID":"string"}, low_memory=False)
    if COL_DATE not in df.columns:
        print(f"[스킵] 날짜 컬럼 없음: {fp.name}"); 
        return
    df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce").dt.normalize()
    df = df.sort_values(COL_DATE)

    site_id = parse_site_id(fp.name)
    rows = []

    for yr in YEARS:
        for o in OFFS:
            for idx, start, end in gen_windows_for_year(yr, o):
                win = df[(df[COL_DATE] >= start) & (df[COL_DATE] <= end)]
                stats = (window_stats(win) if not win.empty else
                        {k: np.nan for k in [
                            "tavg_mean_30d","tmin_mean_30d","tmax_mean_30d",
                            "wind_mean_30d","wind_gust_max_30d","precip_sum_30d","gdd10_sum_30d"
                        ]})
                rows.append({
                    "site_id": site_id,
                    "year": yr,
                    "offset_days": o,
                    "window_idx": idx,
                    "start_date": start.date(),
                    "end_date": end.date(),
                    "days": DAYS_CONST,
                    **stats
                })

    out = pd.DataFrame(rows)

    # 최종 컬럼 순서 고정
    COL_ORDER = [
        "site_id","year","offset_days","window_idx",
        "start_date","end_date","days",
        "tavg_mean_30d","tmin_mean_30d","tmax_mean_30d",
        "wind_mean_30d","wind_gust_max_30d","precip_sum_30d","gdd10_sum_30d"
    ]
    site_id = parse_site_id(fp.name)
    out = out[COL_ORDER]

    out_fname = f"site-{site_id}_WeatherMovingAverage.csv" 
    out_path  = OUT_DIR / out_fname

    out.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.3f")
    print(f"[저장] {out_path} rows={len(out):,}")

def main():
    files = sorted(IN_DIR.glob("*site-*.csv"))  
    if not files:
        print(f"[종료] 입력 없음: {IN_DIR}")
        return
    for fp in files:
        process_one(fp)

if __name__ == "__main__":
    main()

[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32112_58454_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32262_58780_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32550_61437_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32623_61483_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32726_61027_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32746_57621_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32749_58071_WeatherMovingAverage.csv rows=5,400
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/WeatherMovingAverage/site-32896_55900_Weat

### site-관측소 매핑

In [13]:
# build_neighbors_view_tight.py

from pathlib import Path
import numpy as np
import pandas as pd

# ===== 설정 =====
K_NEIGH        = 3                         # site당 이웃 관측소 수
MAX_RADIUS_KM  = None                      # 반경 제한 없으면 None
WEATHER_DIR    = Path("/Users/doyoung-gil/연구실/데이터/기상데이터/ASOS+AWS/GDD")
SITE_MASTER    = Path("/Users/doyoung-gil/연구실/데이터/복숭아/site_master.csv")
OUT_CSV        = Path("/Users/doyoung-gil/연구실/데이터/복숭아/neighbors_view.csv")
DATE_COL       = "일시"
DATE_START     = pd.Timestamp("2010-01-01")
DATE_END       = pd.Timestamp("2025-01-19")

EARTH_R = 6371000.0  # m

def hv(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))  # meters

def read_stations_tight() -> pd.DataFrame:
    # 필요한 컬럼만 로드
    parts = []
    for y in range(DATE_START.year, DATE_END.year + 1):
        p = WEATHER_DIR / f"ASOS_AWS_{y}_with_GDD10.csv"
        if not p.exists():
            continue
        w = pd.read_csv(p, usecols=["지점","지점명","위도","경도", DATE_COL],
                        encoding="utf-8-sig", low_memory=False)
        w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
        w = w[(w[DATE_COL] >= DATE_START) & (w[DATE_COL] <= DATE_END)]
        w[["지점","위도","경도"]] = w[["지점","위도","경도"]].apply(
            pd.to_numeric, errors="coerce"
        )
        parts.append(w)
    if not parts:
        raise SystemExit("ASOS_AWS 원천에서 스테이션을 찾을 수 없습니다.")
    stn = (pd.concat(parts, ignore_index=True)
            .dropna(subset=["지점","위도","경도"])
            .drop_duplicates(subset=["지점"])  # 지점 ID 기준 고유화
            .sort_values("지점"))
    # 컬럼 통일
    stn = stn.rename(columns={"지점명":"관측소 지점명", "위도":"관측소 위도", "경도":"관측소 경도"})
    return stn[["지점","관측소 지점명","관측소 위도","관측소 경도"]].reset_index(drop=True)

def main():
    # 1) site master
    site = pd.read_csv(SITE_MASTER, dtype={"지점ID": str})
    site = (site.rename(columns={"지점ID":"site_id","좌표-위도":"site 위도","좌표-경도":"site 경도"})
                .dropna(subset=["site_id","site 위도","site 경도"]))
    site["site_id"] = site["site_id"].astype(str).str.strip()
    site = site[site["site_id"] != ""].drop_duplicates(subset=["site_id"])

    # 2) stations (tight)
    stn = read_stations_tight()
    stn_lat = stn["관측소 위도"].to_numpy()
    stn_lon = stn["관측소 경도"].to_numpy()

    rows = []
    for sid, slat, slon in site[["site_id","site 위도","site 경도"]].itertuples(index=False):
        d = hv(float(slat), float(slon), stn_lat, stn_lon)  # meters
        if MAX_RADIUS_KM is not None:
            mask = d <= (MAX_RADIUS_KM * 1000.0)
            if not mask.any():   # 반경 내 없으면 전체에서 k-NN
                order = np.argsort(d)[:K_NEIGH]
            else:
                order = np.argsort(d[mask])[:K_NEIGH]
                # mask 인덱스를 원래 인덱스로 복원
                order = np.where(mask)[0][order]
        else:
            order = np.argsort(d)[:K_NEIGH]

        for r, idx in enumerate(order, start=1):
            rows.append({
                "site_id": sid,
                "site 위도": float(slat),
                "site 경도": float(slon),
                "rank" : r,
                "관측소 지점명": stn.at[idx, "관측소 지점명"] if pd.notna(stn.at[idx, "관측소 지점명"]) else "",
                "관측소 위도": float(stn_lat[idx]),
                "관측소 경도": float(stn_lon[idx]),
                "site-관측소 거리": float(d[idx])  # meters
            })

    out = (pd.DataFrame(rows)
            .sort_values(["site_id","rank"])
            .drop_duplicates())  # 동일 거리/중복 방지

    out = out[[
    "site_id", "site 위도", "site 경도",
    "rank",
    "관측소 지점명", "관측소 위도", "관측소 경도",
    "site-관측소 거리"
    ]]
    
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig", float_format="%.6f")
    print(f"[저장] {OUT_CSV}  rows={len(out):,}")

if __name__ == "__main__":
    main()

[저장] /Users/doyoung-gil/연구실/데이터/복숭아/neighbors_view.csv  rows=231


### (R로 fuzzy 돌린 이후) best + 위경도 매핑

In [15]:
# add_and_reorder_coords.py
from pathlib import Path
import pandas as pd

# --- 경로 설정 ---
COORD_CSV = Path("/Users/doyoung-gil/연구실/데이터/복숭아/복숭아 재배 농가 데이터_with_siteid.csv")
BEST_ROOT  = Path("/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best")

# --- 좌표 마스터: '지점ID'를 'site_id'로 통일 후 매핑 ---
coord = (pd.read_csv(COORD_CSV, encoding="utf-8-sig")
            .rename(columns={"지점ID":"site_id", "좌표-위도":"lat", "좌표-경도":"lon"})
            [["site_id","lat","lon"]]
            .drop_duplicates("site_id")
            .set_index("site_id"))

files = sorted(set(BEST_ROOT.rglob("best_*.csv")) | set(BEST_ROOT.rglob("BEST_*.csv")))
print(f"[발견] {len(files)}개")

for f in files:
    df = pd.read_csv(f, encoding="utf-8-sig")

    # 좌표 붙이기
    m = df.merge(coord, on="site_id", how="left", copy=False)
    m["좌표-위도"] = m["lat"]
    m["좌표-경도"] = m["lon"]
    m.drop(columns=["lat","lon"], inplace=True)

    # site_id 바로 뒤로 재배치 (값/열 삭제 없음)
    cols = list(m.columns)
    cols.remove("좌표-위도")
    cols.remove("좌표-경도")
    pos = cols.index("site_id") + 1
    new_cols = cols[:pos] + ["좌표-위도","좌표-경도"] + cols[pos:]

    m[new_cols].to_csv(f, index=False, encoding="utf-8-sig")
    print(f"[저장] {f} rows={len(m)}")

print("[완료]")


[발견] 77개
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32112_58454.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32262_58780.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32550_61437.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32623_61483.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32726_61027.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32746_57621.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32749_58071.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_32896_55900.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_33041_55945.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy/best/best_33044_55808.csv rows=15
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_j

### 모든 지점별 best파일 합치기 + DOY 변환

In [2]:
# merge_best_and_dates_to_doy.py
from pathlib import Path
import pandas as pd

BEST_ROOT = Path("/Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy2/best")
OUT_CSV   = BEST_ROOT / "best_all_sites_DOY.csv"

DATE_COLS = [
    "dormancy_start_date",
    "dormancy_end_date",
    "growing_start_date",
    "growing_end_date",
    "flowering_date",
    "fullbloom_date"
]

def to_doy(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", format="%Y-%m-%d")
    return dt.dt.dayofyear.astype("Int64")

# === 파일 수집: 지점별 best_*.csv만, 통합본(best_all_sites*)은 제외 ===
files = sorted(set(BEST_ROOT.glob("best_*.csv")) | set(BEST_ROOT.glob("BEST_*.csv")))
files = [f for f in files if not f.name.startswith("best_all_sites")]
files = [f for f in files if f.resolve() != OUT_CSV.resolve()]

print(f"[발견] 대상 파일: {len(files)}개")
if not files:
    raise SystemExit("[중단] 병합할 파일이 없습니다. (best_<site>.csv가 있는지 확인)")

dfs = []
for f in files:
    try:
        df = pd.read_csv(f, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(f, encoding="utf-8")

    for col in DATE_COLS:
        if col in df.columns:
            df[col] = to_doy(df[col])

    # 좌표 컬럼 순서 정리(있으면 site_id 뒤로)
    need = ["site_id", "좌표-위도", "좌표-경도"]
    if all(c in df.columns for c in need):
        cols = list(df.columns)
        cols.remove("좌표-위도"); cols.remove("좌표-경도")
        pos = cols.index("site_id") + 1
        cols = cols[:pos] + ["좌표-위도", "좌표-경도"] + cols[pos:]
        df = df[cols]

    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

# (선택) 혹시 모를 중복 키 제거: site_id+year 기준 1행만
if "site_id" in merged.columns and "year" in merged.columns:
    before = len(merged)
    merged = merged.sort_values(["site_id", "year"]).drop_duplicates(["site_id", "year"], keep="first")
    print(f"[중복제거] {before} -> {len(merged)} (site_id,year)")

merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"[저장] {OUT_CSV}  rows={len(merged)}  cols={len(merged.columns)}")
print("[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.")


[발견] 대상 파일: 77개
[중복제거] 1155 -> 1155 (site_id,year)
[저장] /Users/doyoung-gil/연구실/데이터/복숭아/복숭아_joined_samples/Fuzzy2/best/best_all_sites_DOY.csv  rows=1155  cols=13
[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.
